# 🌱 Making It Your Own
### D4BL Tutorial 5 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/05_making_it_your_own.ipynb)

**What you'll learn:** How to adapt D4BL's approach for your own community — bring your own data, customize prompts, and train a model that reflects your priorities.

**Time:** ~20 min (quick mode) / ~35 min (full) | **Prerequisites:** GPU runtime | **Dependencies:** unsloth, transformers, trl

In [ ]:
QUICK_MODE = True  # Set to False for full training

!pip install -q unsloth transformers datasets trl

## The D4BL Methodology is Portable

D4BL's approach isn't just for one organization — it's a framework any community can use:

1. **Center your community** — Who is most impacted? What are their priorities?
2. **Name structural causes** — What historical and policy decisions created these conditions?
3. **Connect to policy** — What specific changes could improve outcomes?
4. **Acknowledge limitations** — What does the data miss? Who isn't counted?
5. **Build power** — How does this analysis serve community organizing and advocacy?

The model you build should reflect *your* community's version of these principles.

## Bring Your Own Data

Replace the sample data below with your community's data. The format is simple: each row needs a metric name, a value, and enough context to generate a meaningful training prompt.

In [ ]:
# === REPLACE THIS WITH YOUR DATA ===
my_data = [
    {"location": "Jefferson County, AL", "metric": "childhood_asthma_rate", "group": "Black children", "value": 14.2, "comparison_group": "white children", "comparison_value": 7.8},
    {"location": "Jefferson County, AL", "metric": "park_access", "group": "Black neighborhoods", "value": 23, "comparison_group": "white neighborhoods", "comparison_value": 67},
]

print(f"Your dataset: {len(my_data)} rows")
for row in my_data:
    ratio = row["value"] / row["comparison_value"] if row["comparison_value"] else 0
    print(f"  {row['metric']} in {row['location']}: {row['group']} = {row['value']}, "
          f"{row['comparison_group']} = {row['comparison_value']} (ratio: {ratio:.1f}x)")

## Customize Your Distillation Prompts

Start from D4BL's system prompt and modify it for your community. Think about:
- What structural factors are most relevant in your area?
- What policy context should the model know about?
- Who is the primary audience for the model's output?

In [ ]:
# === CUSTOMIZE THIS PROMPT ===
MY_SYSTEM_PROMPT = """\
You are an AI assistant trained to support community data analysis \
for [YOUR COMMUNITY/ORGANIZATION NAME].

Core principles:
1. Center [YOUR COMMUNITY] in all analysis and framing.
2. Name structural causes — including [SPECIFIC STRUCTURAL FACTORS IN YOUR AREA].
3. Connect findings to [SPECIFIC POLICY LEVERS RELEVANT TO YOUR CONTEXT].
4. Acknowledge data limitations and gaps in [SPECIFIC DATA CHALLENGES].
5. Make analysis accessible to [YOUR PRIMARY AUDIENCE].

Respond with ONLY valid JSON."""

print("Your custom system prompt:")
print(MY_SYSTEM_PROMPT)
print(f"\nLength: {len(MY_SYSTEM_PROMPT)} chars")

In [ ]:
import json

def build_custom_pair(data_row, system_prompt, response_text):
    """Build a ChatML training pair from your data."""
    metric = data_row["metric"].replace("_", " ")
    location = data_row["location"]
    group = data_row["group"]
    value = data_row["value"]

    user_prompt = f"Explain the {metric} for {group} in {location} ({value})."

    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": response_text},
        ]
    }

example_response = json.dumps({
    "entities": ["Jefferson County", "AL"],
    "search_queries": [
        "childhood asthma rate racial disparity Jefferson County",
        "air quality environmental racism Birmingham Alabama"
    ],
    "data_sources": ["cdc_places", "epa_ejscreen"],
    "community_framing": {
        "detected": True,
        "issue_domain": "health",
        "structural_frame": "environmental_racism"
    }
}, indent=2)

pair = build_custom_pair(my_data[0], MY_SYSTEM_PROMPT, example_response)
print("Your training pair:")
print(json.dumps(pair, indent=2)[:500] + "...")

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048, dtype=None, load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth",
)

training_pairs = []
for row in my_data:
    response = json.dumps({
        "entities": [row["location"]],
        "search_queries": [f"{row['metric']} racial disparity {row['location']}"],
        "data_sources": ["census_acs"],
        "community_framing": {"detected": True, "issue_domain": "health", "structural_frame": "structural_racism"}
    })
    training_pairs.append(build_custom_pair(row, MY_SYSTEM_PROMPT, response))

def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(training_pairs).map(format_example)
print(f"Custom training set: {len(dataset)} examples")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field="text", max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_ratio=0.1, num_train_epochs=1 if QUICK_MODE else 7,
        max_steps=10 if QUICK_MODE else -1, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1, output_dir="my-custom-model", optim="adamw_8bit", seed=42,
    ),
)

print(f"Training ({'QUICK' if QUICK_MODE else 'FULL'} mode)...")
stats = trainer.train()
print(f"Done! Final loss: {stats.training_loss:.4f}")

## What's Next

You've built an equity-focused AI model customized for your community. Here's where to go from here:

### Run locally with Ollama
Export to GGUF format and run on your own computer — no cloud, no API costs:
```python
# Export (from Python)
model.save_pretrained_gguf("my-model", tokenizer, quantization_method="q4_k_m")
```
```bash
# Create Ollama model
ollama create my-community-model -f Modelfile
ollama run my-community-model
```

### Publish to Hugging Face
Share your model with other communities:
```python
model.push_to_hub("your-username/your-model-name")
```

### Contribute back to D4BL
- Share your distillation prompts and methodology adaptations
- Report data gaps or biases you discover
- Join the D4BL community: [d4bl.org](https://d4bl.org)

### Learn more
- [D4BL /learn page](https://d4bl.org/learn) — Interactive visualizations of LoRA, quantization, and the D4BL methodology
- [Slide deck: Building AI That Centers Racial Equity](https://gamma.app/docs/Building-AI-That-Centers-Racial-Equity-m8qd4n13bdtboa1)

## ✏️ Final Exercise

Go end-to-end:
1. Replace `my_data` with real data from your community (even 3-5 rows is enough)
2. Customize `MY_SYSTEM_PROMPT` with your organization's principles
3. Generate training pairs (write the assistant responses yourself — think about what a good analysis looks like)
4. Train the model (QUICK_MODE is fine)
5. Test it: does the output reflect your community's framing?

In [ ]:
# TODO: Your end-to-end pipeline here
# 1. my_data = [...]
# 2. MY_SYSTEM_PROMPT = """..."""
# 3. training_pairs = [build_custom_pair(...) for ...]
# 4. Train (copy the training cell above)
# 5. Test with generate_response() from Notebook 4